# 1. Drifting basics

This is the core notebook. We build a **Drifting Model** from scratch and train it to generate a 2-D spiral in a
single forward pass, then reproduce three of the paper's key experimental findings on a laptop-scale problem.

Reference: [_Generative Modeling via Drifting_](https://arxiv.org/abs/2602.04770), Deng, Li, Li, Du & He (2026).

### The idea in four sentences

1. A generator $f_\theta$ maps noise to samples in one shot; its output distribution is $q_\theta=[f_\theta]_{\#}p_{\boldsymbol{\epsilon}}$.
2. Design a **drifting field** $\mathbf{V}_{p,q}(\mathbf{x})$ that says how each generated sample should move, and that
   **vanishes exactly when $q=p$**.
3. Each training step, compute $\mathbf{V}$ from the current batch, freeze $\mathbf{x}+\mathbf{V}$ as a target, and
   regress the generator onto it.
4. Because training is already iterative, the *training loop itself* performs the transport from noise to data —
   so inference needs **one** network evaluation (1-NFE).

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Anti-symmetry: the one structural requirement

Suppose the field satisfies

$$\mathbf{V}_{p,q}(\mathbf{x}) = -\mathbf{V}_{q,p}(\mathbf{x}) \qquad \forall \mathbf{x}$$

("swapping the two distributions flips the sign"). Then setting $q=p$:

$$\mathbf{V}_{p,p} = -\mathbf{V}_{p,p} \;\;\Longrightarrow\;\; \mathbf{V}_{p,p} = \mathbf{0}$$

That's the whole proof (Proposition 3.1 of the paper). Two things worth being precise about:

* This gives $q=p \Rightarrow \mathbf{V}=\mathbf{0}$. The **converse is not guaranteed in general** — a field can vanish
  without the distributions matching. The paper gives an identifiability argument for its particular kernel construction
  (Appendix C.1, and `THEORY.md` §6) and reports empirically that driving $\|\mathbf{V}\|^2$ down does track sample quality.
* Anti-symmetry is the *only* structural requirement. Everything else — mean shift, Laplace kernels, softmax
  normalisation, feature spaces — is a design choice inside that constraint.

## 2. The field: attraction minus repulsion

Following mean shift (Notebook 0a), define two normalised mean-shift vectors, one from the data and one from the
generator's own samples:

$$\mathbf{V}^{+}_{p}(\mathbf{x}) = \frac{\mathbb{E}_{p}\!\left[k(\mathbf{x},\mathbf{y}^{+})(\mathbf{y}^{+}-\mathbf{x})\right]}{\mathbb{E}_{p}\!\left[k(\mathbf{x},\mathbf{y}^{+})\right]},
\qquad
\mathbf{V}^{-}_{q}(\mathbf{x}) = \frac{\mathbb{E}_{q}\!\left[k(\mathbf{x},\mathbf{y}^{-})(\mathbf{y}^{-}-\mathbf{x})\right]}{\mathbb{E}_{q}\!\left[k(\mathbf{x},\mathbf{y}^{-})\right]}$$

$$\mathbf{V}_{p,q}(\mathbf{x}) := \mathbf{V}^{+}_{p}(\mathbf{x}) - \mathbf{V}^{-}_{q}(\mathbf{x})$$

Combining the two into a single expectation, the $-\mathbf{x}$ terms cancel and we get the paper's Eq. (11):

$$\mathbf{V}_{p,q}(\mathbf{x}) = \mathbb{E}_{\mathbf{y}^{+}\sim p,\;\mathbf{y}^{-}\sim q}\!\left[\tilde{k}(\mathbf{x},\mathbf{y}^{+})\,\tilde{k}(\mathbf{x},\mathbf{y}^{-})\,(\mathbf{y}^{+}-\mathbf{y}^{-})\right]$$

with $\tilde{k}=k/Z$ the normalised kernel. Anti-symmetry is now obvious: swapping $p\leftrightarrow q$ swaps
$\mathbf{y}^{+}\leftrightarrow\mathbf{y}^{-}$, which flips the sign of $(\mathbf{y}^{+}-\mathbf{y}^{-})$ while the
symmetric weight $\tilde k\tilde k$ is unchanged.

The kernel is Laplace with temperature $\tau$:
$$k(\mathbf{x},\mathbf{y})=\exp\!\left(-\tfrac{1}{\tau}\|\mathbf{x}-\mathbf{y}\|\right)$$

## 3. `compute_V` — the paper's Algorithm 2, line by line

In a minibatch we replace the expectations by empirical means. The implementation is a softmax over a distance matrix,
with one extra trick: the paper normalises the logits along **both** axes (over $\mathbf{y}$ *and* over $\mathbf{x}$) and
takes the geometric mean. This is a Sinkhorn-style symmetric normalisation; it makes the affinity matrix behave more
like a doubly-stochastic transport plan and is worth ~0.5 FID (paper Table 11). Crucially it **preserves
anti-symmetry**, because swapping the positive and negative blocks just permutes columns.

In [ ]:
def compute_V(x, y_pos, y_neg=None, tau=0.2, normalize="both"):
    """Drifting field V(x), Algorithm 2 of the paper.

    x     : [N, D]      query points  (= the generated samples)
    y_pos : [P, D]      positives, drawn from p_data
    y_neg : [M, D]      negatives, drawn from q_theta;  None means "reuse x"
    tau   : temperature of the Laplace kernel
    normalize : "both" (row+col softmax, default), "row" (over y only), or "none"
    returns [N, D]
    """
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg:
        y_neg = x

    d_pos = torch.cdist(x, y_pos)                        # [N, P]
    d_neg = torch.cdist(x, y_neg)                        # [N, M]
    if self_neg:                                          # never repel a sample from itself
        d_neg = d_neg + torch.eye(N, device=x.device) * 1e6

    logit = torch.cat([-d_pos / tau, -d_neg / tau], dim=1)   # [N, P+M]

    if normalize == "both":
        A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    elif normalize == "row":
        A = logit.softmax(-1)
    else:
        A = logit.exp()

    P = y_pos.shape[0]
    A_pos, A_neg = A[:, :P], A[:, P:]                    # [N,P], [N,M]

    # weights: each block is scaled by the other block's total mass, which is
    # exactly the product of the two normalised kernels in Eq. (11).
    W_pos = A_pos * A_neg.sum(1, keepdim=True)
    W_neg = A_neg * A_pos.sum(1, keepdim=True)

    return W_pos @ y_pos - W_neg @ y_neg                 # attraction - repulsion

In [ ]:
# --- sanity checks: the two properties the whole method rests on ---
a, b = torch.randn(64, 2), torch.randn(64, 2)

V_pp = compute_V(a, a, a, tau=0.5)                       # positives and negatives identical
print("V vanishes when p == q          :", V_pp.abs().max().item())

V_pq = compute_V(a, b, a, tau=0.5)                       # p = b's cloud, q = a's cloud
V_qp = compute_V(a, a, b, tau=0.5)                       # roles swapped
print("anti-symmetry residual          :", (V_pq + V_qp).abs().max().item())
print("typical drift magnitude         :", V_pq.norm(dim=-1).mean().item())

Both properties hold to float32 precision. Note that `V_pp` is *exactly* zero, not approximately — with the same
empirical set on both sides, the weights are symmetric and the terms cancel identically.

## 4. The training objective

At equilibrium the generator satisfies the fixed point
$f_{\hat\theta}(\boldsymbol{\epsilon}) = f_{\hat\theta}(\boldsymbol{\epsilon}) + \mathbf{V}_{p,q_{\hat\theta}}(f_{\hat\theta}(\boldsymbol{\epsilon}))$.
Turn it into a regression by freezing the right-hand side:

$$\mathcal{L} = \mathbb{E}_{\boldsymbol{\epsilon}}\Big[\big\|\,\underbrace{f_\theta(\boldsymbol{\epsilon})}_{\text{prediction}} - \underbrace{\texttt{sg}\big(f_\theta(\boldsymbol{\epsilon}) + \mathbf{V}_{p,q_\theta}(f_\theta(\boldsymbol{\epsilon}))\big)}_{\text{frozen target}}\big\|^2\Big]$$

Numerically $\mathcal{L}=\mathbb{E}\|\mathbf{V}\|^2$ — the loss *is* the squared drift, so it is a genuine convergence
diagnostic. The gradient handed to the output layer is $-2\mathbf{V}$ (see Notebook 0c). That's the whole algorithm:

In [ ]:
def drifting_loss(x, y_pos, tau=0.2, eta=1.0):
    """Paper's Algorithm 1. `x` must carry gradients; V must not."""
    with torch.no_grad():
        V = eta * compute_V(x, y_pos, None, tau=tau)
        target = x + V                                    # already detached
    return F.mse_loss(x, target)

## 5. Data and generator

A 2-D spiral, and a plain MLP. No time embedding, no U-Net, no noise schedule — a drifting generator has no notion of
time at all.

In [ ]:
def spiral(n, device=device):
    t = torch.sqrt(torch.rand(n, device=device)) * 3.5 * math.pi
    r = 0.35 * t
    xy = torch.stack([r * torch.cos(t), r * torch.sin(t)], -1)
    xy = xy + 0.12 * torch.randn(n, 2, device=device)
    return xy / 3.9                                       # roughly unit scale

class Generator(nn.Module):
    """noise -> sample, single forward pass, no time input"""
    def __init__(self, d_z=2, d_out=2, hidden=256, n_layers=4):
        super().__init__()
        layers, d = [], d_z
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.SiLU()]; d = hidden
        layers += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*layers)
        self.d_z = d_z
    def forward(self, z):
        return self.net(z)
    @torch.no_grad()
    def sample(self, n, device=device):
        return self(torch.randn(n, self.d_z, device=device))

real = spiral(4000)
plt.figure(figsize=(4.2, 4.2))
plt.scatter(real[:, 0].cpu(), real[:, 1].cpu(), s=3, alpha=0.4, c="tab:blue")
plt.title("target distribution  p_data"); plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.show()

## 6. Training

Two implementation details we take from the paper and justify properly in Notebook 3:

* **Multiple temperatures.** One $\tau$ has to choose a single notion of "nearby". The paper sums one drifting loss per
  $\tau\in\{0.02,0.05,0.2\}$ (after normalisation) and reports this beats any single best $\tau$.
* **Drift normalisation.** Each $\mathbf{V}_\tau$ is rescaled to unit RMS before summing, so no temperature dominates,
  and the explicit step size $\eta$ then controls transport speed on a scale that is independent of the data.

In [ ]:
def normalised_drift(V):
    """Rescale V so that E[||V||^2 / D] = 1  (paper eq. 23-25)."""
    lam = V.pow(2).mean().sqrt().clamp_min(1e-8)
    return V / lam

def train_drifting(data_fn, steps=4000, N_neg=256, N_pos=256,
                   taus=(0.05, 0.2, 0.8), eta=0.05, lr=2e-3,
                   d_z=2, snapshot_at=(), seed=0, verbose=True):
    torch.manual_seed(seed)
    g = Generator(d_z=d_z).to(device)
    opt = torch.optim.Adam(g.parameters(), lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    history, raw_drift, snaps = [], [], {}

    for step in range(steps + 1):
        if step in snapshot_at:
            snaps[step] = g.sample(2000).cpu()

        z = torch.randn(N_neg, d_z, device=device)
        x = g(z)                                  # generated samples  (= negatives)
        y_pos = data_fn(N_pos)                    # positives from p_data

        with torch.no_grad():
            V_raw = [compute_V(x, y_pos, None, tau=t) for t in taus]
            # diagnostic: the UN-normalised squared drift, i.e. the real distance from equilibrium
            raw_drift.append(sum(v.pow(2).mean().item() for v in V_raw))
            V = eta * sum(normalised_drift(v) for v in V_raw)
            target = x + V
        loss = F.mse_loss(x, target)

        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0)
        opt.step(); sched.step()
        history.append(loss.item())
        if verbose and step % 1000 == 0:
            print(f"step {step:5d}   loss = {loss.item():.3e}   raw ||V||^2 = {raw_drift[-1]:.3e}")
    return g, history, raw_drift, snaps

snapshot_at = (0, 50, 200, 800, 2000, 4000)
gen, hist, raw, snaps = train_drifting(spiral, snapshot_at=snapshot_at)

## 7. What the training loop actually did to the distribution

This is the paper's Figure 1 / Figure 3: the **pushforward distribution $q_i$ evolving during training**. There is no
inference-time trajectory to plot, because there isn't one — this *is* the trajectory.

In [ ]:
fig, ax = plt.subplots(1, len(snapshot_at) + 2, figsize=(2.05 * (len(snapshot_at) + 2), 2.3))
r = real.cpu()
for a, s in zip(ax, snapshot_at):
    a.scatter(r[:, 0], r[:, 1], s=2, c="tab:blue", alpha=0.20)
    a.scatter(snaps[s][:, 0], snaps[s][:, 1], s=2, c="tab:orange", alpha=0.6)
    a.set_title(f"step {s}", fontsize=9)
    a.set_xlim(-1.3, 1.3); a.set_ylim(-1.3, 1.3)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
ax[-2].semilogy(hist, lw=0.7); ax[-2].set_title("training loss", fontsize=9); ax[-2].grid(alpha=0.3)
ax[-1].semilogy(raw, lw=0.7, c="crimson"); ax[-1].set_title(r"raw $\|V\|^2$", fontsize=9); ax[-1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Two things to notice, and the second is a trap.

**The distribution really is transported by the training loop.** By step 800 the arms of the spiral are there; by
4000 the density profile matches. No inference-time trajectory exists to plot — this *is* the trajectory.

**The training loss is not a convergence diagnostic once you normalise the drift.** Look at the two right-hand panels:
the *training loss* barely moves (2.2e-2 → 1.3e-2) while the *raw* $\|\mathbf{V}\|^2$ falls by almost an order of magnitude.
That is exactly what drift normalisation is *for* — it rescales each $\mathbf{V}_\tau$ to unit RMS, so
$\mathcal{L}\approx\eta^2\cdot\text{const}$ by construction. The identity $\mathcal{L}=\|\mathbf{V}\|^2$ holds for the
field you actually use, and you have deliberately pinned that field's magnitude.

So: **always log the un-normalised $\|\mathbf{V}\|^2$ separately.** That is the number that tracks distance from
equilibrium, and it is the quantity plotted in the paper's Figure 4 (whose toy runs use no drift normalisation).

A related pathology to watch for: if the raw $\|\mathbf{V}\|^2$ is *tiny and flat from step 0*, the model is not
converged — it is **starved**. At initialisation the generated cloud can be so far from the data that every kernel
evaluation sits in its exponential tail, giving a vanishing drift estimate. That is the "flat kernel" problem, and it
is exactly why feature spaces (Notebook 4) matter so much in high dimensions; the paper reports being unable to train
on ImageNet at all without one.

## 8. One-NFE inference

Sampling is a single matrix-multiply chain. Let's time it and compare against a diffusion sampler of the same
architecture cost.

In [ ]:
import time
n = 20000
torch.cuda.synchronize() if device == "cuda" else None
t0 = time.time(); samples = gen.sample(n); torch.cuda.synchronize() if device == "cuda" else None
dt = time.time() - t0
print(f"1-NFE generation of {n} samples: {dt*1e3:.1f} ms   ({n/dt/1e6:.2f} M samples/s)")
print(f"a 200-step diffusion sampler with the same network would cost ~{dt*200*1e3:.0f} ms")

def tv_2d(A, B, bins=24, lim=1.3):
    def h(z):
        H = torch.histogramdd(z.cpu(), bins=[bins, bins],
                              range=[-lim, lim, -lim, lim])[0]
        return H / H.sum()
    return 0.5 * (h(A) - h(B)).abs().sum().item()

ref1, ref2 = spiral(n), spiral(n)
print(f"\nTV(generated, real) = {tv_2d(samples, ref1):.3f}")
print(f"TV(real, real)      = {tv_2d(ref1, ref2):.3f}   <- finite-sample floor")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.7))
for a, (z, t) in zip(ax, [(ref1, "real"), (samples, "drifting, 1 NFE"), (None, "overlay")]):
    if z is None:
        a.scatter(ref1[:, 0].cpu(), ref1[:, 1].cpu(), s=1.5, alpha=0.25, c="tab:blue", label="real")
        a.scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=1.5, alpha=0.25, c="tab:orange", label="gen")
        a.legend(markerscale=6, fontsize=8)
    else:
        a.scatter(z[:, 0].cpu(), z[:, 1].cpu(), s=1.5, alpha=0.25,
                  c="tab:blue" if t == "real" else "tab:orange")
    a.set_title(t); a.set_aspect("equal"); a.set_xlim(-1.3, 1.3); a.set_ylim(-1.3, 1.3)
plt.tight_layout(); plt.show()

## 9. Reproducing the paper's anti-symmetry ablation (Table 1)

The paper runs a *destructive* study: deliberately break anti-symmetry by scaling attraction or repulsion, and watch
FID explode from 8.46 to 177. Let's reproduce the mechanism on a bimodal target where the failure is visible by eye.
We use the explicit $\mathbf{V}^{+}-\mathbf{V}^{-}$ form so the two halves can be scaled independently.

In [ ]:
def mean_shift(x, S, tau, exclude_self=False):
    d = torch.cdist(x, S)
    if exclude_self:
        d = d + torch.eye(x.shape[0], device=x.device) * 1e6
    return (-d / tau).softmax(-1) @ S - x

def V_split(x, y_pos, tau, a_pos=1.0, a_neg=1.0):
    Vp = mean_shift(x, y_pos, tau)
    Vn = mean_shift(x, x, tau, exclude_self=True)
    return a_pos * Vp - a_neg * Vn

def bimodal(n, sep=1.0, sd=0.15, device=device):
    idx = torch.randint(0, 2, (n,), device=device)
    c = torch.tensor([[-sep, 0.0], [sep, 0.0]], device=device)
    return c[idx] + sd * torch.randn(n, 2, device=device)

def train_split(a_pos, a_neg, init_bias=None, steps=2500, tau=0.25, eta=0.1, seed=1):
    torch.manual_seed(seed)
    g = Generator().to(device)
    if init_bias is not None:                     # start the model collapsed on one mode
        with torch.no_grad():
            g.net[-1].weight.mul_(0.05)
            g.net[-1].bias.copy_(torch.tensor(init_bias, device=device))
    opt = torch.optim.Adam(g.parameters(), 2e-3)
    for _ in range(steps):
        x = g(torch.randn(256, 2, device=device))
        with torch.no_grad():
            V = eta * V_split(x, bimodal(256), tau, a_pos, a_neg)
        loss = F.mse_loss(x, (x + V).detach())
        opt.zero_grad(); loss.backward(); opt.step()
    return g.sample(8000).cpu()

cases = [("anti-symmetric (default)", 1.0, 1.0),
         ("1.5x attraction",          1.5, 1.0),
         ("1.5x repulsion",           1.0, 1.5),
         ("2.0x attraction",          2.0, 1.0),
         ("attraction only",          1.0, 0.0)]

tgt = bimodal(8000).cpu()
print(f"{'case':26s} {'std_x':>7s} {'std_y':>7s} {'left mode share':>16s}")
print(f"{'target p_data':26s} {tgt[:,0].std():7.3f} {tgt[:,1].std():7.3f} "
      f"{(tgt[:,0]<0).float().mean():16.3f}")
outs = {}
for name, ap, an in cases:
    s = train_split(ap, an); outs[name] = s
    print(f"{name:26s} {s[:,0].std():7.3f} {s[:,1].std():7.3f} {(s[:,0]<0).float().mean():16.3f}")

In [ ]:
fig, ax = plt.subplots(1, len(cases) + 1, figsize=(2.1 * (len(cases) + 1), 2.4))
ax[0].scatter(tgt[:, 0], tgt[:, 1], s=2, c="tab:blue", alpha=0.4); ax[0].set_title("p_data", fontsize=9)
for a, (name, _, _) in zip(ax[1:], cases):
    s = outs[name]
    a.scatter(s[:, 0], s[:, 1], s=2, c="tab:orange", alpha=0.4)
    a.set_title(name, fontsize=8)
for a in ax:
    a.set_xlim(-1.7, 1.7); a.set_ylim(-0.9, 0.9); a.set_aspect("equal")
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Exactly the paper's result, in miniature:

* **anti-symmetric** — recovers $p$: both modes, right widths, 50/50 balance.
* **excess attraction** — total collapse to a point (`std = 0.000`). Attraction wins, nothing balances it.
* **excess repulsion** — survives, but over-disperses (`std_y` roughly doubles). Degradation, not catastrophe.
* **attraction only** — mean shift with no repulsion is mode seeking. It collapses, as it must.

Excess attraction is far more destructive than excess repulsion — the same asymmetry the paper reports
(FID 86 for $2\times$ attraction vs 113 for $2\times$ repulsion, but 177 for attraction-only). The lesson is
structural: **the repulsion term is what makes drifting a generative model rather than a mode finder.**

## 10. Robustness to initialisation (paper Figure 3)

The paper's most striking toy result: even when $q$ starts fully **collapsed onto one mode**, drifting escapes and
recovers the other. The mechanism is that repulsion from the crowded region beats attraction locally, so samples keep
moving until some of them come under the influence of the other mode.

In [ ]:
inits = {"between the modes": [0.0, 0.0],
         "far from both":     [0.0, 2.5],
         "collapsed on one":  [1.0, 0.0]}

fig, ax = plt.subplots(2, len(inits), figsize=(3.0 * len(inits), 5.2))
for j, (name, b) in enumerate(inits.items()):
    for i, (an, tag) in enumerate([(1.0, "anti-symmetric"), (0.0, "attraction only")]):
        s = train_split(1.0, an, init_bias=b)
        ax[i, j].scatter(tgt[:, 0], tgt[:, 1], s=2, c="tab:blue", alpha=0.25)
        ax[i, j].scatter(s[:, 0], s[:, 1], s=2, c="tab:orange", alpha=0.5)
        ax[i, j].set_title(f"init: {name}\n{tag}   (left share {(s[:,0]<0).float().mean():.2f})",
                           fontsize=8)
        ax[i, j].set_xlim(-1.7, 1.7); ax[i, j].set_ylim(-0.9, 0.9)
        ax[i, j].set_aspect("equal"); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
plt.tight_layout(); plt.show()

Top row (anti-symmetric): all three initialisations recover both modes with ~50/50 balance. Bottom row
(attraction only): the collapsed initialisation **stays collapsed on the wrong mode** — 0.00 share for the left mode.

This is the concrete sense in which drifting is robust to mode collapse without adversarial training. A GAN can
collapse because the discriminator's signal is only "be more realistic"; the drifting field's repulsion term makes
"be somewhere your own samples aren't" a first-class part of the objective.

## 11. Recap: the entire method

```python
for step in range(n_steps):
    eps   = randn(N, C)                        # prior noise
    x     = f_theta(eps)                       # generated samples == negatives
    y_pos = sample_data(P)                     # positives

    with no_grad():
        V = sum(normalise(compute_V(x, y_pos, tau=t)) for t in taus) * eta

    loss = mse(x, (x + V).detach())            # regress onto the drifted target
    loss.backward(); opt.step()
```

Sampling: `f_theta(randn(n, C))`. One call. No schedule, no solver, no timestep.

### What we established

| claim | how we checked it |
|---|---|
| $\mathbf{V}$ vanishes at $q=p$ | `V_pp.abs().max() == 0` |
| $\mathbf{V}_{p,q}=-\mathbf{V}_{q,p}$ | residual $\sim10^{-8}$ |
| $\mathcal{L}=\|\mathbf{V}\|^2$ | Notebook 0c, exact to 1e-9 |
| training moves the pushforward | the six-panel evolution in §7 |
| anti-symmetry is required | Table-1 ablation reproduced in §9 |
| robust to collapsed init | §10 |
| 1-NFE inference | §8 |

### Where to go next

* [2. Anatomy of the Drifting Field](<2. Anatomy of the Drifting Field.ipynb>) — dissect $\mathbf{V}$: normalisation
  variants, temperature, positives/negatives allocation, and what the field looks like as a picture.
* [3. Practical Drifting](<3. Practical Drifting.ipynb>) — the recipe that makes it work at scale.